# Toy BNN — paper figures

Loads per-split `.pt` files written by `sazz.scripts.uci_bnn` and produces
1. A metrics table for one (dataset, split) — for quick inspection.
2. The single-dataset 2×N predictive panel — for inspection.
3. **The paper figure**: a 4×5 grid of predictive bands across all four toy datasets and all five samplers (NUTS + four PDMPs). This is the figure that goes in the toy-BNN subsection.
4. **Appendix figure**: the matching 4×5 grid of epistemic-std-vs-x panels.

## Setup

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")
from sazz.scripts.bnns.toy_bnn import (
    build_target
)

torch.set_default_dtype(torch.float64)

# ---- Pick what to inspect ----
DATASET     = "hernandez"   # hernandez / gap / sharp / multiscale
SPLIT_ID    = 0
RESULTS_DIR = Path("results/toy_bnns")

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
print(f"Looking in {split_dir}")
print(f"  found: {sorted(p.name for p in split_dir.glob('*.pt'))}")


## Load all sampler runs

Each `.pt` is a self-contained payload: thinned samples, x_ref, layer_sizes,
metrics, etc. We load them into a dict keyed by sampler name.

In [ ]:
def load_runs(split_dir: Path) -> dict[str, dict]:
    runs = {}
    for pt_path in sorted(split_dir.glob("*.pt")):
        name = pt_path.stem
        runs[name] = torch.load(pt_path, weights_only=False)
    return runs

runs = load_runs(split_dir)
print(f"Loaded {len(runs)} samplers: {list(runs)}")

# Quick peek at one payload
example_sampler = list(runs)[0]
print(f"\nKeys in {example_sampler}.pt: {list(runs[example_sampler])}")


## Metrics summary